# Domain adaptation in classical machine learning

## Research question and hypotheses

For Logistic Regression, linear SVM and Random Forest, when does importance weighting or correlation alignment (CORAL) improve or worsen held-out target macro F1 relative to source-only training under changes in gas sensor collection batch, smartphone participants and product review category, and what accuracy and computation-time trade-offs accompany these changes?

For each transfer pair and classifier, define the paired effect as `delta = macro_F1(adapted) - macro_F1(source_only)` on identical target test IDs. **H1:** importance weighting yields a positive delta. **H2:** CORAL yields a positive delta. Each hypothesis is assessed separately for every pair and classifier; zero or negative effects are valid outcomes. Weighting is expected to depend on source–target overlap, and CORAL on useful mean/covariance alignment; these are diagnostic interpretations, not guaranteed improvements or additional selection criteria.

## Domains and prediction tasks

| Shift | Transfer | Pairs |
|---|---|---|
| Time | Gas batch 1 to 2-10 | 9 |
| People | HAR training to test | 1 |
| Category | Books to Electronics | 1 |

### Time: UCI Gas Sensor Array Drift

**Labelled source:** All batch 1 rows; predict one of six gases

**Unlabelled adaptation:** For each batch 2–10 separately, shuffle sorted IDs with the run seed and take the first floor(N/2) rows

**Held-out evaluation:** Remaining rows of that same later batch

**Transfer pairs:** 9: batch 1 → 2, 3, 4, 5, 6, 7, 8, 9, 10

### People: UCI HAR Using Smartphones

**Labelled source:** All 21 official training participants; predict one of six activities

**Unlabelled adaptation:** Shuffle the nine sorted official test participant IDs with the seed; take four whole participants

**Held-out evaluation:** Other five whole official test participants; keep each person's windows together

**Transfer pairs:** 1 participant-transfer design

### Category: Amazon Multi-Domain Sentiment

**Labelled source:** Books positive and negative reviews from `processed_acl.tar.gz`; predict sentiment

**Unlabelled adaptation:** Electronics unlabelled reviews

**Held-out evaluation:** Electronics positive and negative reviews

**Transfer pairs:** 1: Books → Electronics

There are **11 transfer pairs**. Restart from Gas batch 1 for every later batch. Roles are disjoint; assign target roles without task labels. Amazon release roles stay fixed across seeds, subject to a feature-only duplicate audit: source takes priority over target test, which takes priority over adaptation. Strip `#label#` from every Amazon input dictionary and exclude domain, path, polarity filename and rating from predictors. Batch and participant identifiers are grouping metadata, not predictors.

## Registered comparison matrix

Apply every row below to every transfer pair and seed. All rows predict the same held-out IDs within that pair and seed.

| Configuration | Classifier | Training condition |
|---|---|---|
| LR-SO | Logistic Regression | Source only |
| LR-IW | Logistic Regression | Importance weighting |
| LR-CORAL | Logistic Regression | CORAL |
| SVM-SO | Linear SVM (`LinearSVC`) | Source only |
| SVM-IW | Linear SVM (`LinearSVC`) | Importance weighting |
| SVM-CORAL | Linear SVM (`LinearSVC`) | CORAL |
| RF-SO | Random Forest | Source only |
| RF-IW | Random Forest | Importance weighting |
| RF-CORAL | Random Forest | CORAL |

The majority-class control is separate from these nine main configurations.

**Separate control - Source majority:** Predict the most frequent source task label; break ties by sorted class ID

**Control rows per transfer pair and seed:** 1, independently of classifier.

### Information available for fitting

**Source only:** Source features and source task labels

**Importance weighting:** Source features/task labels plus unlabelled adaptation features; source/adaptation domain membership trains the domain classifier

**CORAL:** Source features/task labels plus unlabelled adaptation features for mean/covariance alignment

**Majority control:** Source task-label counts only

No method uses adaptation task labels. Held-out target features are used only for frozen transformation and prediction; held-out task labels are read only by the final evaluator after predictions are frozen. Target scores never select settings.

## Shared representation, settings and ablation

Use seeds **11, 23, 37, 53, 71**. Reuse each split manifest and ordered test-ID list across methods. Seeds control eligible splits, source folds, randomized reduction and estimators; Amazon's released task roles do not change.

Fit ordinary preprocessing on source training rows only, inside each source-validation fold: imputation when needed and standardization for Gas/HAR; source dictionary, TF-IDF, truncated SVD and standardization for Amazon. All branches, including Random Forest, share the same starting representation. For Amazon use `k = min(100, n_source_rows - 1, n_source_features - 1)` within each fit.

Tune each classifier by source-only three-fold macro-F1 validation, with participant-disjoint stratified group folds for HAR and shuffled stratified folds otherwise. Reuse the selected settings in all three branches. Logistic Regression and LinearSVC use `C = [0.1, 1, 10]`; Random Forest uses 200 trees, square-root feature sampling and `max_depth = [12, None]`. Resolve exact tuning ties by the first candidate. Use `class_weight=None` and one CPU thread per estimator. Source-only tuning is a limitation of this comparison.

For importance weighting, use an unweighted logistic domain classifier (`C=1`) and seeded three-fold out-of-fold source probabilities. Clamp target probability `p` to `[1e-6, 1-1e-6]`; calculate `(p / (1-p)) * (n_source_fit / n_adapt_fit)` using domain-training counts for that fold. Cap raw weights at 10, then normalize to mean one. Record overlap/weight diagnostics and effective sample size. For CORAL, align source means and covariances to adaptation data in float64, adding `1e-3 I` to each covariance; leave target features in their original shared coordinates.

**Registered ablation:** remove only the upper weight cap, retaining probability safeguards, mean normalization, source settings and matched splits. Compare uncapped minus capped importance weighting separately for all three classifiers. Freeze main and ablation predictions before opening target test labels.

**Planned inventory:** `11 × 5 × 3 × 3 = 495` main rows; `11 × 5 = 55` majority-control rows; `11 × 5 × 3 = 165` additional uncapped-weight rows. These are planned counts, not observed results. Exclude pilots, tuning and retries from final totals; check configuration identities as well as counts.

## Outcomes and interpretation

**Primary:** target macro F1 over the sorted classes with positive true support in the original test split, using zero for an undefined class F1. Use the same label set for all methods and bootstrap draws within that split. Report per-class support and full-vocabulary confusion matrices.

**Secondary:** target accuracy and wall-clock seconds on the same hosted Colab CPU. Total compute time is shared preprocessing fit/transform + adaptation fit/transform + classifier fit + prediction. Charge measured shared stages to each branch that reuses them. Exclude downloading, file I/O, tuning, scoring and bootstrapping; report tuning time separately. Positive accuracy differences indicate improvement; positive time differences indicate added cost.

Compare each adapted method minus its matching source-only classifier, by transfer pair and seed. Report per-seed effects and five-seed mean ± sample SD for scores and paired effects. Keep transfer pairs visible so nine Gas transfers do not dominate the other datasets.

For **seed 11 only**, use a paired 95% percentile bootstrap interval for macro-F1 delta: 2,000 draws with resampling seed 2026 and identical resampled units for both methods. Resample whole participants for HAR, unique retained reviews for Amazon and measurements within each Gas target batch. HAR has only five evaluation clusters; Gas intervals are conditional on batch and assume independent measurements, so temporal dependence may make them too narrow. Repeated splits and shared source data are dependent.

Predeclare this descriptive rule: an interval entirely above zero supports a **gain**; entirely below zero supports a **loss / negative transfer**; an interval containing or touching zero is **inconclusive**, not evidence of equivalence. No practical-effect threshold or multiplicity-adjusted claim is prespecified. The seed-11 interval does not describe uncertainty in the five-seed mean; report their signs separately and disclose disagreement. Apply the same interval rule to uncapped-minus-capped ablation effects. No experimental result is asserted here.

## Related work: evidence matrix

The ten papers below cover weighting foundations, domain-adaptation theory, classical CORAL, transfer evaluation and the three application areas. Each evidence statement refers to its cited paper and original protocol; it is not an outcome of the present study. Section, table and figure locations support checking the paraphrases.

| ID | Reference | Theme |
|---|---|---|
| R1 | [Shimodaira (2000)](https://stat.sys.i.kyoto-u.ac.jp/wp-content/uploads/2017/07/1998-Shimodaira-rm712-covariate-shift.pdf) | Weighted likelihood |
| R2 | [Sugiyama et al. (2007)](https://www.jmlr.org/papers/volume8/sugiyama07a/sugiyama07a.pdf) | Weighted validation |
| R3 | [Huang et al. (2007)](https://www.cs.cmu.edu/~arthurg/papers/HuaSmoGreBorSch07.pdf) | Kernel mean matching |
| R4 | [Bickel et al. (2009)](https://www.jmlr.org/papers/volume10/bickel09a/bickel09a.pdf) | Domain weighting |
| R5 | [Ben-David et al. (2010)](https://doi.org/10.1007/s10994-009-5152-4) | Adaptation theory |
| R6 | [Saenko et al. (2010)](https://doi.org/10.1007/978-3-642-15561-1_16) | Visual transfer |
| R7 | [Sun et al. (2016)](https://doi.org/10.1609/aaai.v30i1.10306) | Classical CORAL |
| R8 | [Blitzer et al. (2007)](https://aclanthology.org/P07-1056.pdf) | Review sentiment |
| R9 | [Anguita et al. (2013)](https://www.esann.org/sites/default/files/proceedings/legacy/es2013-84.pdf) | Smartphone HAR |
| R10 | [Liu and Tang (2013)](https://www.mdpi.com/1424-8220/13/7/9160) | Gas sensor drift |

The detailed records below preserve every field of the evidence matrix.

### R1 - Shimodaira (2000)

H. Shimodaira (2000), [Improving predictive inference under covariate shift by weighting the log-likelihood function](https://doi.org/10.1016/S0378-3758(00)00115-4); [author's 1998 preprint](https://stat.sys.i.kyoto-u.ac.jp/wp-content/uploads/2017/07/1998-Shimodaira-rm712-covariate-shift.pdf)

**Task and original data assignment:** Polynomial regression with source covariates from N(0.5, 0.5²), future covariates from N(0, 0.3²), and the same noisy cubic response.

**Method:** Density-ratio weighted likelihood / least squares.

**Published evidence and location:** Preprint §2/Fig. 1 and §3 address future-distribution loss under misspecification; §4/Lemma 4 favors uniform weighting asymptotically under correct specification.

**Limitation for this study:** Evidence locations refer to the accessible 1998 preprint, not the paywalled journal text. Regression results with known distributions do not establish classification gains here.

### R2 - Sugiyama et al. (2007)

M. Sugiyama, M. Krauledat, K.-R. Müller (2007), [Covariate Shift Adaptation by Importance Weighted Cross Validation](https://www.jmlr.org/papers/volume8/sugiyama07a/sugiyama07a.pdf)

**Task and original data assignment:** EEG binary classification: 14 datasets from five subjects; earlier-session labelled training, later-session unlabelled first half and evaluated second half.

**Method:** Importance-weighted cross-validation (IWCV) selects adaptive weighted LDA.

**Published evidence and location:** §5, Tables 1–2, pp. 997–998: IWCV-selected models improve over ordinary LDA in five cases and worsen in one.

**Limitation for this study:** Oracle OPT results are separate. Density-ratio assumptions and model-selection variance matter; our tuning is source-only and classifiers differ.

### R3 - Huang et al. (2007)

J. Huang, A. J. Smola, A. Gretton, K. M. Borgwardt, B. Schölkopf (2007 proceedings; NIPS 2006), [Correcting Sample Selection Bias by Unlabeled Data](https://www.cs.cmu.edu/~arthurg/papers/HuaSmoGreBorSch07.pdf)

**Task and original data assignment:** Toy regression and biased classification samples; source task labels and unlabelled target covariates supply the fit. §4.1 uses 100 source and 100 target observations.

**Method:** Kernel mean matching (KMM) estimates weights through constrained feature-mean matching.

**Published evidence and location:** §3.2/Lemma 4: the sampling bound worsens with the upper density-ratio bound, even with correct population weights.

**Limitation for this study:** The bound concerns feature means, not macro F1. KMM is not our logistic domain classifier; weight constraints motivate diagnostics without validating our cap of 10.

### R4 - Bickel et al. (2009)

S. Bickel, M. Brückner, T. Scheffer (2009), [Discriminative Learning Under Covariate Shift](https://www.jmlr.org/papers/volume10/bickel09a/bickel09a.pdf)

**Task and original data assignment:** Spam, Cora text and landmine classification. Cora has 812 pre-1996 source papers and 1,285 later target papers; source years provide tuning data.

**Method:** Domain-probability weighting, with integrated and two-stage discriminative models.

**Published evidence and location:** §10/Fig. 1 reports 15% relative reduction in Cora's 1−AUC risk for integrated models at 1,024 target examples.

**Limitation for this study:** This is not accuracy-point gain or a result for our two-stage implementation. Published tuning/metrics differ; our out-of-fold probabilities, normalization and cap are declared choices.

### R5 - Ben-David et al. (2010)

S. Ben-David, J. Blitzer, K. Crammer, A. Kulesza, F. Pereira, J. W. Vaughan (2010), [A theory of learning from different domains](https://doi.org/10.1007/s10994-009-5152-4); [full paper](https://escholarship.org/content/qt2nv1j9sc/qt2nv1j9sc.pdf)

**Task and original data assignment:** Classification theory for labelled source and unlabelled target samples; additional settings consider limited target labels.

**Method:** Generalization bounds involving hypothesis-class domain divergence.

**Published evidence and location:** Theorem 2, pp. 157–158: target error also depends on source error, sample complexity and ideal joint-hypothesis error.

**Limitation for this study:** Reducing marginal divergence alone cannot guarantee good target classification. This is not a macro-F1 guarantee or an empirical validation on our datasets.

### R6 - Saenko et al. (2010)

K. Saenko, B. Kulis, M. Fritz, T. Darrell (2010), [Adapting Visual Category Models to New Domains](https://doi.org/10.1007/978-3-642-15561-1_16); [full paper](https://people.bu.edu/bkulis/pubs/saenko_eccv_2010.pdf)

**Task and original data assignment:** Object recognition across Amazon product photographs, DSLR and webcam images; the same-category experiment uses three labelled target examples per category.

**Method:** Supervised cross-domain metric learning.

**Published evidence and location:** §5/Table 1 reports improved cross-domain recognition with learned transformations, with results differing by direction and method.

**Limitation for this study:** Target supervision and visual features differ from our unlabelled adaptation setting. Here “Amazon” denotes product images, not sentiment reviews.

### R7 - Sun et al. (2016)

B. Sun, J. Feng, K. Saenko (2016), [Return of Frustratingly Easy Domain Adaptation](https://doi.org/10.1609/aaai.v30i1.10306); [full paper](https://cdn.aaai.org/ojs/10306/10306-13-13834-1-2-20201228.pdf)

**Task and original data assignment:** Object and review classification with labelled source/unlabelled target data. The Office protocol adapts using target test features (§4.1).

**Method:** Classical CORAL whitens/recolors source features; source-validated linear SVM.

**Published evidence and location:** Table 5, p. 2064: Books→Electronics accuracy is 76.3% with CORAL versus 74.7% without adaptation.

**Limitation for this study:** Top-400-word features and 20 splits differ from ours. Algorithm 1 uses ridge 1; our ridge 1e-3, explicit mean alignment and disjoint target roles are implementation differences.

### R8 - Blitzer et al. (2007)

J. Blitzer, M. Dredze, F. Pereira (2007), [Biographies, Bollywood, Boom-boxes and Blenders: Domain Adaptation for Sentiment Classification](https://aclanthology.org/P07-1056.pdf)

**Task and original data assignment:** Books, DVDs, Electronics and Kitchen sentiment; 1,600 labelled training and 400 test reviews per domain, plus unlabelled data.

**Method:** Structural correspondence learning with mutual-information pivots (SCL-MI), linear Huber-loss classifier.

**Published evidence and location:** §4/Fig. 1, pp. 442–443: DVDs→Books accuracy rises from 72.8% for baseline to 79.7% for SCL-MI.

**Limitation for this study:** Different method and split. The 46% transfer-error reduction in §5/Table 3 includes 50 labelled target examples; it must not be attributed to label-free adaptation.

### R9 - Anguita et al. (2013)

D. Anguita, A. Ghio, L. Oneto, X. Parra, J. L. Reyes-Ortiz (2013), [A Public Domain Dataset for Human Activity Recognition Using Smartphones](https://www.esann.org/sites/default/files/proceedings/legacy/es2013-84.pdf)

**Task and original data assignment:** Six activities from 30 volunteers with 561 features; published 70%/30% volunteer train/test partition.

**Method:** Gaussian-kernel one-versus-all SVM with ten-fold hyperparameter validation.

**Published evidence and location:** §3/Table 4, pp. 440–441: 96% overall accuracy on 2,947 test patterns.

**Limitation for this study:** A dataset/baseline result, not an adaptation gain. Our linear SVM and four/five target-participant roles differ; published accuracy is not an expected score for this study.

### R10 - Liu and Tang (2013)

H. Liu, Z. Tang (2013), [Metal Oxide Gas Sensor Drift Compensation Using a Dynamic Classifier Ensemble Based on Fitting](https://www.mdpi.com/1424-8220/13/7/9160)

**Task and original data assignment:** Ten chronological gas batches; five analytes after excluding toluene. Fig. 3 trains on batches 1–5, 1–6, 1–7 or 1–8, then tests later batches.

**Method:** RBF-SVM ensembles with time-dependent classifier weights fitted from labelled early batches.

**Published evidence and location:** §4/Fig. 3 reports DWF outperforming the tested alternatives while performance still deteriorates over time.

**Limitation for this study:** Multiple labelled batches exceed our batch-1 source budget. These are classifier weights, not source-example importance weights. §5 notes slow weight search and fitting-function sensitivity.

This matrix contains **10 paper entries**. The [official HAR record](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones) confirms participant-based roles, and the [Amazon authors' release notes](https://www.cs.jhu.edu/~mdredze/datasets/sentiment/) warn that released positive/negative files need not reproduce the paper's splits. These dataset records are supporting provenance, not additional paper entries.

## Research gap and study contribution

The reviewed evidence supports conditional adaptation benefits. Weighting depends on the source–target relation and estimation stability (R1–R4); domain-adaptation theory also retains a joint-prediction-error term (R5). Thus a smaller feature mismatch does not by itself establish better target macro F1. These are published theoretical or empirical observations, not results from the present study.

Prior evaluations use different data budgets, representations and outcomes. Supervised visual transfer uses target labels (R6); classical CORAL includes an Office setting that adapts on target test features and a review experiment with a different representation (R7). Sentiment, smartphone and gas studies use their own splits, classifiers and label budgets (R8–R10). Their numerical scores cannot be used as direct baselines for the protocol registered here.

**Specific research gap and contribution.** Within this reviewed set, there is no single evaluation of the registered combination: Logistic Regression, linear SVM and Random Forest × source-only, importance weighting and classical CORAL across nine Gas batch transfers, one HAR participant-transfer design and Books→Electronics, with disjoint adaptation/evaluation roles, matched target IDs and shared source-fitted representations. This is a bounded comparison gap inferred from these ten papers, not an exhaustive novelty claim about all literature.

Holding classifiers, source-only tuning, metrics and data access consistent makes it possible to examine whether an adaptation effect changes with classifier and shift type. The study will report paired target macro-F1 changes, accuracy and staged CPU cost across five seeds, a separate source-majority control, weight/overlap diagnostics and an uncapped-weight ablation. Per-transfer reporting and the registered uncertainty procedure keep improvements, losses and inconclusive effects visible.

The hypotheses remain **H1: importance weighting improves paired target macro F1** and **H2: CORAL improves paired target macro F1**, assessed separately by classifier and transfer pair. Positive published results motivate these tests; they do not verify them. Source-only tuning is a deliberate fairness constraint and a limitation, rather than a replication of every paper's model-selection procedure. The implementation uses classical CORAL with the declared mean alignment and ridge setting, not a neural-network CORAL loss. No target score has been observed or used to revise the design.